<a href="https://colab.research.google.com/github/sharinawas-dev/Laptop-price-Predication/blob/main/RAG_Laptop_Troubleshoot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#RAG
    #Data
    #Chunking
    #Loading
    #Vector Store-Chromos,FAISS
    #Query:
    #reterive chunk Query
    #Augument to system prompt
    #Generate the output

In [2]:
from google.colab import userdata
Gemini_API_Key=userdata.get('Gemini_API_Key')
Grok_API_Key=userdata.get('Grok_API_Key')

In [16]:
from google.colab import files
uploaded=files.upload()

Saving dell_latitude_5420_troubleshooting_demo (1).pdf to dell_latitude_5420_troubleshooting_demo (1).pdf


In [17]:
!pip install -q langchain-community LLM langchain-text-splitters langchain-groq langchain-google-genai
!pip install -q pypdf fastapi uvicorn langchain-core faiss-cpu

In [18]:
from langchain_community.document_loaders import PyPDFLoader

In [19]:
loader=PyPDFLoader("dell_latitude_5420_troubleshooting_demo.pdf")
pages=loader.load()

In [20]:
print(pages)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-13T19:52:30+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-13T19:52:30+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'dell_latitude_5420_troubleshooting_demo.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content="Classroom RAG demo \x7f Dell Latitude 5420 troubleshooting\nPage 1\n Laptop Troubleshooting Knowledge Base\n Dell Latitude 5420 — RAG classroom demonstration document\nImportant: This is a fictionalized training handout created for a classroom RAG demonstration. It is not an official Dell\nservice manual. Troubleshooting steps are general, conservative guidance. Always check the device's official\ndocumentation and warranty terms before repair.\n1. Identify the device and collect symptoms\nThis guide covers the Dell Latitude 5420 laptop as a demonstration model. Before troub

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=700,chunk_overlap=100)
chunks=text_splitter.split_documents(pages)
print("Number of Chunks:",len(chunks))
print("\n First Chunks:\n")
print(chunks[2].page_content)

Number of Chunks: 14

 First Chunks:

compatible charger if available.
 Leave the laptop connected to power for 15–30 minutes, then try powering it on again.
 Look for charging or status LEDs and note any blink pattern. Do not guess the meaning of a blink code; verify it in the
official documentation for the exact configuration.
 If there is a burning smell, liquid damage, unusual heat, or a swollen battery, disconnect power if safe and stop using
the device. Arrange professional service.
Escalate when: There is still no response with a known-good compatible power source, or physical damage is
suspected. Do not open the laptop unless authorized and trained.
3. Laptop powers on but screen is black


In [22]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",api_key=Gemini_API_Key
)

In [23]:
from langchain_community.vectorstores import FAISS

In [24]:
db=FAISS.from_documents(chunks,embeddings)
print("Faiss Database Created!")

Faiss Database Created!


In [25]:
db

In [26]:
embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x787261365550>, model='models/gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [27]:
from langchain_groq import ChatGroq

In [28]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=Grok_API_Key,temperature=0

)
print(llm)

metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.4.8'}} profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x7872442def90> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7872442dfcb0> model_name='openai/gpt-oss-20b' temperature=1e-08 model_kwargs={} groq_api_key=SecretStr('**********')


In [30]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
You are a laptop troubleshooting assistant.

Answer only the context provided below.

If the answer is not available in the context,say:
"I could not find the answer in teh manual."

Context
{context}

Question
{question}

Give a clear,easy to understand answer.
""")

In [47]:
def ask_question(question):
    #Step1 : Search Faiss from relevant chunk/Query
    result=db.similarity_search(question, k=4)
    #Step2: combine the reterive text
    context="\n\n".join(
        doc.page_content for doc in result
    )
    #Step3 : prepare the prompt
    message=prompt.format_messages(
        context=context,
        question=question
    )
    #Step4: Ask the LLM
    response=llm.invoke(message)

    #Step5: Display the answer
    print("Answer:\n")
    print(response.content)

    #step6: Show the reterived sources page
    print("\nSOURCES:\n")
    for doc in result:
        print("page:",doc.metadata.get("page",0)+1)
        print(doc.page_content[:200])
        print("-"*40)
    return response.content





In [48]:
rep=ask_question("Laptop is  slow")
print(rep)

Answer:

**If your laptop is running slow, try these quick steps:**

1. **Restart** the laptop – a fresh start often clears temporary hiccups.  
2. **Check disk space** – make sure you have enough free space; delete or move large files if needed.  
3. **Close unused programs** – open Task Manager (Ctrl + Shift + Esc) and end any apps that you’re not using.  
4. **Look for high‑resource processes** – in Task Manager, see if any program is using a lot of CPU, memory, or disk. End or update those if possible.  
5. **Keep it cool** – place the laptop on a hard, flat surface. Avoid using it on beds, pillows, or other soft surfaces that block airflow.  
6. **Clean the vents** – gently remove dust from the vents with a can of compressed air. Do not insert objects into the vents or open the chassis unless you’re trained.  
7. **Check the fan** – if the fan sounds like grinding or the laptop keeps shutting down from heat, schedule a service appointment.  

If the slowdown persists after these s